# Optimización de Red de Oficinas

## Objetivos de Aprendizaje

En este notebook, aprenderás a:

1. **Analizar rendimiento de oficinas** con métricas multi-dimensionales
2. **Usar funciones geoespaciales** (`ST_DISTANCE`, `ST_MAKEPOINT`) para análisis de cobertura
3. **Clasificar oficinas** por rentabilidad con `SNOWFLAKE.ML.CLASSIFICATION`
4. **Calcular solapamiento geográfico** entre oficinas cercanas
5. **Generar recomendaciones** de apertura/cierre con `CORTEX.COMPLETE()`

---

## Lo Que Construirás

Un **sistema de optimización de red de oficinas**:
- Scoring de rentabilidad por oficina
- Análisis geoespacial de cobertura y solapamiento
- Recomendaciones de apertura, cierre o redimensionamiento
- Dashboard de gestión de red

**Valor de Negocio**: Optimizar costes de red manteniendo cobertura de clientes.

---

## Funcionalidades Clave

- `ST_DISTANCE()`, `ST_MAKEPOINT()` — Análisis geoespacial
- `SNOWFLAKE.ML.CLASSIFICATION` — Predicción de rentabilidad
- `CORTEX.COMPLETE()` — Recomendaciones estratégicas
- Window functions — Rankings y comparativas

¡Comencemos!

---

## Paso 1: Configuración del Entorno

### Optimización de Red Bancaria

**Objetivo**: Evaluar la rentabilidad y cobertura geográfica de cada oficina para tomar decisiones de apertura/cierre.

In [ ]:
CREATE DATABASE IF NOT EXISTS RED_OFICINAS_DB;
CREATE SCHEMA IF NOT EXISTS RED_OFICINAS_DB.PUBLIC;
USE SCHEMA RED_OFICINAS_DB.PUBLIC;

CREATE WAREHOUSE IF NOT EXISTS COMPUTE_WH 
    WITH WAREHOUSE_SIZE = 'SMALL'
    AUTO_SUSPEND = 60 AUTO_RESUME = TRUE;
USE WAREHOUSE COMPUTE_WH;

---

## Paso 2: Generar Datos de Red de Oficinas

### Qué Vamos a Crear

- **200 oficinas** con coordenadas reales de ciudades españolas
- **Métricas de rendimiento**: clientes, transacciones, ingresos, costes
- **Datos demográficos** de zona por código postal

In [ ]:
-- Oficinas con coordenadas
CREATE OR REPLACE TABLE OFICINAS (
    oficina_id VARCHAR(10) PRIMARY KEY,
    nombre VARCHAR(100),
    ciudad VARCHAR(50),
    lat FLOAT,
    lon FLOAT,
    superficie_m2 INTEGER,
    num_empleados INTEGER,
    fecha_apertura DATE,
    tipo VARCHAR(20),
    coste_mensual DECIMAL(10,2),
    clientes_asignados INTEGER,
    transacciones_mensuales INTEGER,
    ingresos_mensuales DECIMAL(10,2),
    es_rentable BOOLEAN
);

INSERT INTO OFICINAS
WITH ciudades AS (
    SELECT column1 AS ciudad, column2 AS lat_base, column3 AS lon_base FROM VALUES
    ('Madrid',40.4168,-3.7038),('Barcelona',41.3851,2.1734),('Valencia',39.4699,-0.3763),
    ('Sevilla',37.3891,-5.9845),('Bilbao',43.2630,-2.9350),('Zaragoza',41.6488,-0.8891),
    ('Málaga',36.7213,-4.4214),('Murcia',37.9922,-1.1307),('Palma',39.5696,2.6502),
    ('Las Palmas',28.1235,-15.4363)
)
SELECT
    'OFI' || LPAD(SEQ4()::STRING, 4, '0'),
    'Oficina ' || c.ciudad || ' ' || SEQ4(),
    c.ciudad,
    c.lat_base + (UNIFORM(-50, 50, RANDOM()) / 1000.0),
    c.lon_base + (UNIFORM(-50, 50, RANDOM()) / 1000.0),
    UNIFORM(80, 500, RANDOM()),
    UNIFORM(3, 25, RANDOM()),
    DATEADD('day', -UNIFORM(365, 15000, RANDOM()), CURRENT_DATE()),
    ARRAY_CONSTRUCT('Flagship','Estándar','Express','Digital')[UNIFORM(0,3,RANDOM())]::VARCHAR,
    ROUND(UNIFORM(8000, 80000, RANDOM()), 2),
    UNIFORM(500, 15000, RANDOM()),
    UNIFORM(200, 5000, RANDOM()),
    ROUND(UNIFORM(5000, 120000, RANDOM()), 2),
    CASE WHEN UNIFORM(0,100,RANDOM()) < 70 THEN TRUE ELSE FALSE END
FROM TABLE(GENERATOR(ROWCOUNT => 20)) CROSS JOIN ciudades c
WHERE SEQ4() <= 200;

SELECT ciudad, COUNT(*) AS oficinas, ROUND(AVG(ingresos_mensuales),0) AS ing_medio
FROM OFICINAS GROUP BY 1 ORDER BY 2 DESC;

---

## Paso 3: Análisis Geoespacial

### Distancias entre Oficinas

Usamos `ST_DISTANCE` para calcular solapamiento geográfico.

In [ ]:
-- Calcular distancias entre oficinas de la misma ciudad
CREATE OR REPLACE TABLE DISTANCIAS_OFICINAS AS
SELECT
    a.oficina_id AS oficina_a,
    b.oficina_id AS oficina_b,
    a.ciudad,
    ROUND(ST_DISTANCE(
        ST_MAKEPOINT(a.lon, a.lat),
        ST_MAKEPOINT(b.lon, b.lat)
    ) / 1000, 2) AS distancia_km
FROM OFICINAS a
JOIN OFICINAS b ON a.ciudad = b.ciudad AND a.oficina_id < b.oficina_id
WHERE ST_DISTANCE(ST_MAKEPOINT(a.lon, a.lat), ST_MAKEPOINT(b.lon, b.lat)) < 5000;

-- Oficinas con solapamiento (<2km)
SELECT
    oficina_a, oficina_b, ciudad, distancia_km
FROM DISTANCIAS_OFICINAS
WHERE distancia_km < 2
ORDER BY distancia_km
LIMIT 20;

---

## Paso 4: Modelo de Rentabilidad

### Clasificar Oficinas

Entrenamos un modelo que predice si una oficina será rentable basándose en ubicación, tamaño y métricas operativas.

In [ ]:
CREATE OR REPLACE TABLE FEATURES_OFICINAS AS
SELECT
    oficina_id,
    superficie_m2::FLOAT,
    num_empleados::FLOAT,
    DATEDIFF('year', fecha_apertura, CURRENT_DATE())::FLOAT AS antiguedad_anos,
    coste_mensual::FLOAT,
    clientes_asignados::FLOAT,
    transacciones_mensuales::FLOAT,
    ingresos_mensuales::FLOAT,
    (ingresos_mensuales / NULLIF(coste_mensual, 0))::FLOAT AS ratio_ingreso_coste,
    (clientes_asignados / NULLIF(num_empleados, 0))::FLOAT AS clientes_por_empleado,
    CASE tipo WHEN 'Flagship' THEN 3 WHEN 'Estándar' THEN 2 WHEN 'Express' THEN 1 ELSE 0 END::FLOAT AS nivel_tipo,
    es_rentable
FROM OFICINAS;

CREATE OR REPLACE TABLE TRAIN_OFI AS SELECT * FROM FEATURES_OFICINAS SAMPLE (80);
CREATE OR REPLACE TABLE TEST_OFI AS SELECT * FROM FEATURES_OFICINAS WHERE oficina_id NOT IN (SELECT oficina_id FROM TRAIN_OFI);

CREATE OR REPLACE SNOWFLAKE.ML.CLASSIFICATION predictor_rentabilidad(
    INPUT_DATA => SYSTEM$REFERENCE('TABLE', 'TRAIN_OFI'),
    TARGET_COLNAME => 'ES_RENTABLE'
);

In [ ]:
CALL predictor_rentabilidad!SHOW_EVALUATION_METRICS();
CALL predictor_rentabilidad!SHOW_FEATURE_IMPORTANCE();

---

## Paso 5: Recomendaciones IA

Generamos recomendaciones estratégicas para cada oficina.

In [ ]:
CREATE OR REPLACE TABLE RECOMENDACIONES_RED AS
SELECT
    o.oficina_id, o.nombre, o.ciudad, o.tipo,
    o.ingresos_mensuales, o.coste_mensual,
    ROUND(o.ingresos_mensuales / NULLIF(o.coste_mensual, 0), 2) AS ratio_ic,
    SNOWFLAKE.CORTEX.COMPLETE('mistral-large',
        'Eres un consultor de banca retail. Recomienda una acción (Mantener/Redimensionar/Cerrar/Digitalizar) para esta oficina y justifica en 2 líneas:\n' ||
        'Oficina: ' || o.nombre || '\n' ||
        'Tipo: ' || o.tipo || '\n' ||
        'Ingresos: ' || o.ingresos_mensuales || '€/mes\n' ||
        'Costes: ' || o.coste_mensual || '€/mes\n' ||
        'Ratio I/C: ' || ROUND(o.ingresos_mensuales / NULLIF(o.coste_mensual, 0), 2) || '\n' ||
        'Clientes: ' || o.clientes_asignados || '\n' ||
        'Empleados: ' || o.num_empleados || '\n' ||
        'Responde en español.'
    ) AS recomendacion_ia
FROM OFICINAS o
SAMPLE (30 ROWS);

SELECT oficina_id, nombre, ratio_ic, recomendacion_ia FROM RECOMENDACIONES_RED LIMIT 5;

---

## Paso 6: Dashboard de Gestión de Red

In [ ]:
import streamlit as st
import pandas as pd
from snowflake.snowpark.context import get_active_session

session = get_active_session()

st.title('Optimización de Red de Oficinas')
st.markdown('### Análisis de Rentabilidad y Cobertura')

df = session.sql('SELECT * FROM OFICINAS').to_pandas()

c1, c2, c3 = st.columns(3)
c1.metric('Total Oficinas', len(df))
c2.metric('Rentables', len(df[df['ES_RENTABLE']]))
c3.metric('No Rentables', len(df[~df['ES_RENTABLE']]))

st.markdown('---')
st.subheader('Por Ciudad')
df_city = df.groupby('CIUDAD').agg({'OFICINA_ID':'count','INGRESOS_MENSUALES':'mean'}).round(0)
df_city.columns = ['Oficinas','Ingreso Medio']
st.dataframe(df_city)

st.markdown('---')
st.subheader('Solapamiento Geográfico (<2km)')
df_dist = session.sql('SELECT oficina_a, oficina_b, ciudad, distancia_km FROM DISTANCIAS_OFICINAS WHERE distancia_km < 2 ORDER BY distancia_km LIMIT 15').to_pandas()
st.dataframe(df_dist)

st.markdown('---')
st.subheader('Recomendaciones IA')
df_rec = session.sql('SELECT nombre, ratio_ic, recomendacion_ia FROM RECOMENDACIONES_RED ORDER BY ratio_ic LIMIT 10').to_pandas()
st.dataframe(df_rec)

---

## Paso 7: Limpieza (Opcional)

In [ ]:
-- Descomentar para limpiar

-- DROP DATABASE IF EXISTS RED_OFICINAS_DB;